# BabyLM Persian Fine-tuning

This notebook adapts fine-tuning setup to train **eng-baseline-small** on Persian text using **fas-baseline-small** tokenizer.

## Setup

In [ ]:
# Install dependencies
!pip install -q -U transformers datasets accelerate huggingface-hub tensorboard
!pip install -q torch  # Make sure latest PyTorch

import torch, transformers, datasets
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Datasets:     {datasets.__version__}")
print(f"CUDA avail:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"VRAM:         {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

PyTorch:      2.10.0+cu128
Transformers: 5.6.2
Datasets:     4.8.5
CUDA avail:   True
GPU:          NVIDIA A100-SXM4-40GB
VRAM:         42.4 GB


In [ ]:
# Setup Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/babylm_persian"
except ImportError:
    # Local setup
    PROJECT_DIR = "./babylm_persian"

import os
for sub in ["checkpoints", "results", "logs", "data"]:
    os.makedirs(f"{PROJECT_DIR}/{sub}", exist_ok=True)

print(f"Project dir: {PROJECT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/babylm_persian


In [ ]:
# Login to HuggingFace Hub
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Load Models & Tokenizers

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_IDS = {
    "eng_base": "BabyLM-community/eng-baseline-small",
    "fas_base": "BabyLM-community/fas-baseline-small",
    "eng_fas": "BabyLM-community/eng-fas-baseline-small",
}

# Load eng model and fas tokenizer for this experiment
print("Loading eng-baseline-small model...")
eng_model = AutoModelForCausalLM.from_pretrained(
    MODEL_IDS["eng_base"],
    torch_dtype=torch.float32,
).to("cuda")

print("Loading fas-baseline-small tokenizer...")
fas_tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS["fas_base"])

if fas_tokenizer.pad_token is None:
    fas_tokenizer.pad_token = fas_tokenizer.eos_token

n_params = sum(p.numel() for p in eng_model.parameters()) / 1e6
print(f"Model params: {n_params:.1f}M")
print(f"Tokenizer vocab: {len(fas_tokenizer)}")

Loading eng-baseline-small model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

Loading fas-baseline-small tokenizer...
Model params: 17.1M
Tokenizer vocab: 8192


In [ ]:
# Resize embeddings if needed
if len(fas_tokenizer) != eng_model.config.vocab_size:
    print(f"Resizing embeddings: {eng_model.config.vocab_size} -> {len(fas_tokenizer)}")
    eng_model.resize_token_embeddings(len(fas_tokenizer))

## Load & Prepare Data

In [ ]:
from datasets import load_dataset, concatenate_datasets
import random

# Load Persian corpus from BabyLM
BABYLM_FAS_ID = "BabyLM-community/babylm-fas"
print(f"Loading {BABYLM_FAS_ID}...")
train_ds = load_dataset(BABYLM_FAS_ID, split="train")
print(f"Dataset size: {len(train_ds)}")
print(f"Columns: {train_ds.column_names}")
print(f"\nSample:")
print(train_ds[0]["text"][:300])

Loading BabyLM-community/babylm-fas...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/149M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/150M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/164M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset size: 217776
Columns: ['text', 'doc-id', 'category', 'data-source', 'script', 'age-estimate', 'license', 'misc', 'num-tokens', 'language']

Sample:
دسته ی بیل او بسیار کوتاه بود.


## Pack Dataset into Blocks

In [ ]:
def pack_dataset(ds, tok, block_size=512, text_col="text"):
    """Tokenize and pack into fixed-length blocks.
    Uses return_overflowing_tokens=True to keep all tokens (no data loss).
    """
    def tok_fn(batch):
        out = tok(
            batch[text_col],
            truncation=True,
            max_length=block_size,
            return_overflowing_tokens=True,  # Keep all tokens
            padding=False,
            add_special_tokens=True,
        )
        out.pop("overflow_to_sample_mapping", None)
        return out

    tokenized = ds.map(tok_fn, batched=True, remove_columns=ds.column_names)

    def group(ex):
        concat = {k: sum(ex[k], []) for k in ex}
        total = (len(concat["input_ids"]) // block_size) * block_size
        return {k: [v[i:i + block_size] for i in range(0, total, block_size)]
                for k, v in concat.items()}

    return tokenized.map(group, batched=True)


print("Packing dataset...")
train_blocks = pack_dataset(train_ds, fas_tokenizer, block_size=512)
print(f"Packed: {len(train_blocks)} blocks (~{len(train_blocks)*512/1e6:.1f}M tokens)")

Packing dataset...


Map:   0%|          | 0/217776 [00:00<?, ? examples/s]

Map:   0%|          | 0/356773 [00:00<?, ? examples/s]

Packed: 248908 blocks (~127.4M tokens)


In [ ]:
from google.colab import drive

os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)

# After you pack the datasets:
print("Saving tokenized datasets to Drive...")

# Save Persian blocks
train_blocks.save_to_disk(f'{PROJECT_DIR}/data/fas_blocks')
print(f"✓ Persian blocks saved to {PROJECT_DIR}/data/fas_blocks")

# Optional: Save config
import json
config = {
    "fas_blocks_count": len(train_blocks),
    "block_size": 512,
    "tokenizers": {
        "fas": "BabyLM-community/fas-baseline-small"
    }
}
with open(f'{PROJECT_DIR}/data/config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(f"✓ Config saved")

Saving tokenized datasets to Drive...


Saving the dataset (0/1 shards):   0%|          | 0/97656 [00:00<?, ? examples/s]

✓ Persian blocks saved to /content/drive/MyDrive/babylm_persian/data/fas_blocks
✓ Config saved


In [ ]:
from datasets import load_from_disk
from google.colab import drive
import json

# Load blocks
fas_blocks = load_from_disk(f'{PROJECT_DIR}/data/fas_blocks')

# Load config
with open(f'{PROJECT_DIR}/data/config.json') as f:
    config = json.load(f)

print(f"✓ Loaded {len(fas_blocks)} Persian blocks")


✓ Loaded 97656 Persian blocks


In [ ]:
# Optional: cap training data
MAX_TRAIN_TOKENS = 50_000_000
max_blocks = MAX_TRAIN_TOKENS // 512
if len(fas_blocks) > max_blocks:
    print(f"Capping to {max_blocks} blocks (~{MAX_TRAIN_TOKENS/1e6:.1f}M tokens)")
    train_blocks = train_blocks.shuffle(seed=42).select(range(max_blocks))

print(f"Final: {len(fas_blocks)} blocks")

Final: 97656 blocks


In [ ]:
# Split into train (99%) and eval (1%)
split = fas_blocks.train_test_split(test_size=0.01, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Eval samples: {len(eval_dataset)}")

Training samples: 96679
Eval samples: 977


In [ ]:
train_dataset.save_to_disk(f"{PROJECT_DIR}/data/fas_train")
eval_dataset.save_to_disk(f"{PROJECT_DIR}/data/fas_test")

Saving the dataset (0/1 shards):   0%|          | 0/96679 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/977 [00:00<?, ? examples/s]

## Training Setup

**Hyperparameters from original BabyLM setup:**
- learning_rate: 0.0001
- train_batch_size: 64
- eval_batch_size: 8
- optimizer: AdamW (betas=(0.9, 0.999), eps=1e-8)
- lr_scheduler_type: linear
- num_epochs: 5

In [ ]:
from transformers import (
    Trainer, TrainingArguments, DataCollatorForLanguageModeling
)
import inspect

# Create training arguments
use_bf16 = torch.cuda.is_bf16_supported()
print(f"Using {'bf16' if use_bf16 else 'fp16'} precision")

out_dir = f"{PROJECT_DIR}/checkpoints"

training_args = TrainingArguments(
    output_dir=out_dir,
    num_train_epochs=5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=8,
    learning_rate=0.0001,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    bf16=use_bf16,
    fp16=(not use_bf16),
    dataloader_num_workers=4,
    optim="adamw_torch",
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=1e-8,
    seed=42,
    report_to=["tensorboard"],
)

print("Training arguments created")
print(f"Total steps: {len(train_dataset) * 5 // 64}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Using bf16 precision
Training arguments created
Total steps: 7553


In [ ]:
# Create data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=fas_tokenizer, mlm=False)

# Create trainer
# Handle both "tokenizer" and "processing_class" parameter names
ta_params = set(inspect.signature(Trainer).parameters)
tok_kwarg = "processing_class" if "processing_class" in ta_params else "tokenizer"

trainer = Trainer(
    model=eng_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    **{tok_kwarg: fas_tokenizer},
)

print("Trainer created")

Trainer created


## Train!

In [ ]:
print("Starting training...")
print(f"Expected time: {len(train_dataset) * 5 / 64 / 100:.1f}h (very rough estimate)")
train_result = trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Starting training...
Expected time: 75.5h (very rough estimate)


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
500,5.874348,5.618551
1000,5.007592,4.805470
1500,4.659851,4.468081
2000,4.467003,4.288631
2500,4.342069,4.169424
3000,4.254669,4.089093
3500,4.193782,4.028737
4000,4.148618,3.983736
4500,4.104890,3.943212
5000,4.060743,3.916267


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [ ]:
# Save final model and tokenizer
final_dir = f"{PROJECT_DIR}/final_model_eng_fas_noreplay"
trainer.save_model(final_dir)
fas_tokenizer.save_pretrained(final_dir)

print(f"\nModel saved to {final_dir}")
print(f"Files: {os.listdir(final_dir)}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to /content/drive/MyDrive/babylm_persian/final_model_eng_fas_noreplay
Files: ['checkpoints', 'results', 'config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin']


## Generation Test

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load fine-tuned model
final_dir = f"{PROJECT_DIR}/final_model_eng_fas_noreplay"
tok = AutoTokenizer.from_pretrained(final_dir)
mdl = AutoModelForCausalLM.from_pretrained(final_dir, torch_dtype=torch.float16).to("cuda")
mdl.eval()

def generate(prompt, max_new=50):
    ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")
    with torch.no_grad():
        out = mdl.generate(
            ids,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0], skip_special_tokens=True)

# Test with Persian prompts
prompts = [
    "یک روز",
    "دختر کوچکی بود",
    "در جنگل عمیق",
]

for prompt in prompts:
    result = generate(prompt)
    print(f"Prompt: {prompt}")
    print(f"Generated: {result}")
    print()

Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

Prompt: یک روز
Generated: یک روز جمعه در مورد روز چهارشنبه بود که به دست آمده است تا از ابتدای سال جاری، با اشاره به اینکه این امر گفت: به طور کامل یک میلیون نفر دیگر نیز به دست آورده است و اگر در سال گذشته با توجه به آن ها رسیده اند و

Prompt: دختر کوچکی بود
Generated: دختر کوچکی بود که ما باید آن را داشته باشیم.
وی با بیان اینکه در این زمینه، تصریح کرد: اگر ما به عنوان یک کشور ما باید به خاطر یک طرف هم دارد و تا چه زمانی که ما برای حضور در اختیار مردم قرار دهیم، گفت:

Prompt: در جنگل عمیق
Generated: در جنگل عمیق و آبزی، به عنوان یک روش می تواند در جنگل عمیق باشد.
وی اضافه کرد: جنگل عمیق با کودکان در حال انجام است که یکی از آنها، قاعدان وجود دارد و حتی از این رو هیچ گونه ای برای تنش یا آب



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load fine-tuned model
final_dir = f"{PROJECT_DIR}/final_model_eng_fas_noreplay"
tok = AutoTokenizer.from_pretrained(final_dir)
mdl = AutoModelForCausalLM.from_pretrained(final_dir, torch_dtype=torch.float16).to("cuda")
mdl.eval()

def generate(prompt, max_new=50):
    ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")
    with torch.no_grad():
        out = mdl.generate(
            ids,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0], skip_special_tokens=True)

# Test with Persian prompts
prompts = [
  "One Day",
  "There was a young girl",
  "In a forest"
]

for prompt in prompts:
    result = generate(prompt)
    print(f"Prompt: {prompt}")
    print(f"Generated: {result}")
    print()

Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

Prompt: One Day
Generated: One Daygrinians asmory as f.com (ISM) و همچنین به طور کلی، در سال های اخیر یک بار دیگر در سال ها و چند بار دیگر بیشتر از سه بار دیگر نیز در سال هایی

Prompt: There was a young girl
Generated: There was a young girldires. Rartonnen
در سال ۱۹۸۵۳ به مدت زمان بازرسند و از جمله این که تعداد و تعداد آن با کیفیت های بیشتر است، می تواند در مدت زمان و کار را کسب کند.


Prompt: In a forest
Generated: In a forest conssicd) در آن زمان با هدف های مختلف برای کمک به درمان کمک می کند.
از آنجا که استفاده از مواد غذایی استفاده نکنید، استفاده کنید. این روغن ها در محیط زیست مانند میوه ها و کیفیت خاص است.



## Save Results

In [ ]:
import json

results = {
    "model_id": "BabyLM-community/eng-baseline-small",
    "tokenizer_id": "BabyLM-community/fas-baseline-small",
    "dataset_id": "BabyLM-community/babylm-fas",
    "training_loss": float(train_result.training_loss),
    "eval_loss": float(eval_result.get("eval_loss", float("nan"))),
    "hyperparameters": {
        "learning_rate": 0.0001,
        "train_batch_size": 64,
        "eval_batch_size": 8,
        "num_epochs": 5,
        "optimizer": "adamw_torch",
        "optimizer_betas": [0.9, 0.999],
        "optimizer_epsilon": 1e-8,
        "lr_scheduler_type": "linear",
        "seed": 42,
    }
}

results_path = f"{PROJECT_DIR}/results/metrics.json"
os.makedirs(os.path.dirname(results_path), exist_ok=True)
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {results_path}")
print(json.dumps(results, indent=2))

Results saved to /content/drive/MyDrive/babylm_persian/results/metrics.json
{
  "model_id": "BabyLM-community/eng-baseline-small",
  "tokenizer_id": "BabyLM-community/fas-baseline-small",
  "dataset_id": "BabyLM-community/babylm-fas",
  "training_loss": 4.45329569672055,
  "eval_loss": 3.842007875442505,
  "hyperparameters": {
    "learning_rate": 0.0001,
    "train_batch_size": 64,
    "eval_batch_size": 8,
    "num_epochs": 5,
    "optimizer": "adamw_torch",
    "optimizer_betas": [
      0.9,
      0.999
    ],
    "optimizer_epsilon": 1e-08,
    "lr_scheduler_type": "linear",
    "seed": 42
  }
}


## EVAL

In [ ]:
!pip install sacrebleu
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=6c27323b8a5b0e6497967d2e2c8a62de1811b26c395f9e36c4473f42f74a0c4c
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
import json, math, time
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from sacrebleu import corpus_bleu, corpus_chrf
from rouge_score import rouge_scorer

with open(f"{PROJECT_DIR}/eval_pairs.json", encoding="utf-8") as f:
    eval_pairs = json.load(f)
print(f"Loaded {len(eval_pairs)} eval pairs")

# Safety: ensure MODEL_IDS key exists if cells ran out of order
if "eng_fas" not in MODEL_IDS:
    MODEL_IDS["eng_fas"] = "BabyLM-community/eng-fas-baseline-small"

VARIANT_PATHS = {
    "A_eng_to_fas": f"{PROJECT_DIR}/final_model/",
    # "B_fas_to_eng": f"{PROJECT_DIR}/checkpoints/variant_B/final",
    "C_joint":      MODEL_IDS["eng_fas"],
}

STRICT_LANG_MATRIX = True
LANG_MATRIX = {
    "A_eng_to_fas": ["fa", "en"],
    # "B_fas_to_eng": ["en", "fa"],
    "C_joint":      ["en", "fa"],
}

def load_variant(path):
    tok = AutoTokenizer.from_pretrained(path)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    mdl = AutoModelForCausalLM.from_pretrained(path, torch_dtype=dtype).to("cuda")
    mdl.eval()
    return mdl, tok


Loaded 496 eval pairs


In [ ]:
# ============================================================
# evaluation
#
@torch.no_grad()
def generate_continuations_batched(model, tok, prompts, max_new=80, batch_size=16,
                                   decoding="sampling"):
    outs = []
    model.config.pad_token_id = tok.pad_token_id
    orig_side = tok.padding_side
    tok.padding_side = "left"

    if decoding == "sampling":
        gen_kwargs = dict(
            do_sample=True, temperature=0.8, top_p=0.9,
            repetition_penalty=1.2, no_repeat_ngram_size=3,
        )
    elif decoding == "beam":
        gen_kwargs = dict(
            do_sample=False, num_beams=4, length_penalty=1.0,
            no_repeat_ngram_size=3, early_stopping=True,
        )
    else:
        raise ValueError(decoding)

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        enc = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=256).to("cuda")
        prompt_len = enc.input_ids.shape[1]
        gen = model.generate(**enc, max_new_tokens=max_new,
                             pad_token_id=tok.pad_token_id, **gen_kwargs)
        new_tokens = gen[:, prompt_len:]
        decoded = tok.batch_decode(new_tokens, skip_special_tokens=True)
        outs.extend(d.strip() for d in decoded)
    tok.padding_side = orig_side
    return outs


@torch.no_grad()
def compute_perplexity_and_bpb(model, tok, texts, batch_size=8, max_length=512):
    """Returns (perplexity, bits_per_byte). BPB is tokenizer-invariant."""
    nlls = 0.0
    n_tokens = 0
    n_bytes = 0
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        n_bytes += sum(len(t.encode("utf-8")) for t in batch)
        enc = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=max_length).to("cuda")
        labels = enc.input_ids.clone()
        labels[enc.attention_mask == 0] = -100
        out = model(**enc, labels=labels)
        valid = (enc.attention_mask.sum(dim=1) - 1).clamp(min=0).sum().item()
        if valid > 0:
            nlls += out.loss.item() * valid
            n_tokens += valid
    if n_tokens == 0 or n_bytes == 0:
        return float("nan"), float("nan")
    ppl = math.exp(nlls / n_tokens)
    # bits per byte: total_nats / (ln(2) * total_bytes)
    bpb = (nlls) / (math.log(2) * n_bytes)
    return ppl, bpb


rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def score(preds, refs):
    # Use sacrebleu's Persian-aware tokenizer if available; default is fine
    bleu = corpus_bleu(preds, [refs]).score
    chrf = corpus_chrf(preds, [refs]).score
    rouge_l = sum(rouge.score(r, p)["rougeL"].fmeasure
                  for p, r in zip(preds, refs)) / len(preds)
    return {"BLEU": bleu, "chrF": chrf, "ROUGE-L": rouge_l * 100}


results = []
sample_outputs = {}

for v_name, v_path in VARIANT_PATHS.items():
    print(f"\n=== {v_name} ===")
    model, tok = load_variant(v_path)
    langs = LANG_MATRIX[v_name] if STRICT_LANG_MATRIX else ["en", "fa"]
    for lang in langs:
        prompts = [p[f"{lang}_prompt"] for p in eval_pairs]
        refs    = [p[f"{lang}_continuation"] for p in eval_pairs]

        # Score with beam search (stable), sample for qualitative display
        preds_beam    = generate_continuations_batched(model, tok, prompts, decoding="beam")
        preds_sampled = generate_continuations_batched(model, tok, prompts[:10], decoding="sampling")

        m = score(preds_beam, refs)
        ppl, bpb = compute_perplexity_and_bpb(model, tok, refs)
        m["PPL"] = ppl
        m["BPB"] = bpb
        results.append({"variant": v_name, "lang": lang, **m})
        print(f"  [{lang}] BLEU={m['BLEU']:.2f}  chrF={m['chrF']:.2f}  "
              f"ROUGE-L={m['ROUGE-L']:.2f}  PPL={ppl:.2f}  BPB={bpb:.3f}")

        sample_outputs[f"{v_name}_{lang}"] = [
            {"prompt": prompts[i], "reference": refs[i],
             "generation_beam": preds_beam[i],
             "generation_sampled": preds_sampled[i] if i < len(preds_sampled) else None}
            for i in range(min(10, len(preds_beam)))
        ]
    del model, tok
    torch.cuda.empty_cache()



=== A_eng_to_fas ===


Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

  [fa] BLEU=0.47  chrF=12.51  ROUGE-L=0.02  PPL=61.78  BPB=1.029
  [en] BLEU=0.01  chrF=7.33  ROUGE-L=8.07  PPL=27.00  BPB=2.479

=== C_joint ===


config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/119M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

  [en] BLEU=1.79  chrF=14.64  ROUGE-L=15.46  PPL=29.33  BPB=1.167
  [fa] BLEU=0.98  chrF=14.15  ROUGE-L=0.02  PPL=56.62  BPB=0.936


In [ ]:
df = pd.DataFrame(results)
print("\n=== Final Results ===")
print(df.pivot(index="variant", columns="lang",
               values=["BLEU", "ROUGE-L", "PPL"]).round(2))

df.to_csv(f"{PROJECT_DIR}/results/metrics.csv", index=False)
with open(f"{PROJECT_DIR}/results/samples.json", "w", encoding="utf-8") as f:
    json.dump(sample_outputs, f, ensure_ascii=False, indent=2)

print(f"\nResults saved to {PROJECT_DIR}/results/")


=== Final Results ===
              BLEU       ROUGE-L          PPL       
lang            en    fa      en    fa     en     fa
variant                                             
A_eng_to_fas  0.01  0.47    8.07  0.02  27.00  61.78
C_joint       1.79  0.98   15.46  0.02  29.33  56.62

Results saved to /content/drive/MyDrive/babylm_persian/results/


In [ ]:
# MultiBLiMP: grammatical-minimal-pair accuracy.
# Hardened: correct Persian code (fas), per-row error isolation, length
# truncation to max_position_embeddings, NaN/inf guards, sync CUDA errors.

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # make CUDA errors synchronous & truthful

from datasets import load_dataset
import torch, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

MULTIBLIMP_ID = "jumelet/multiblimp"
# MultiBLiMP on HF uses 'fas' for Persian (not the 'pes' macrolanguage code)
MBLIMP_LANGS = {"en": "eng", "fa": "fas"}


def load_multiblimp_subset(lang_code):
    ds = load_dataset(MULTIBLIMP_ID, lang_code, split="train")
    print(f"  [{lang_code}] {len(ds)} pairs | columns: {ds.column_names[:6]}...")
    return ds


@torch.no_grad()
def sentence_logprob(model, tok, text):
    """Sum of log-probs of every token in `text`. Returns None if the row
    is degenerate (empty, OOV, too long, or produces NaN/inf loss)."""
    if not text or not text.strip():
        return None

    max_pos = getattr(model.config, "max_position_embeddings", 512)
    # Leave a little headroom under the position limit
    max_len = min(max_pos, 1024) - 2

    enc = tok(text, return_tensors="pt", truncation=True, max_length=max_len)
    ids = enc.input_ids.to("cuda")
    if ids.size(1) < 2:
        return None

    vocab_size = model.get_input_embeddings().num_embeddings
    if ids.max().item() >= vocab_size or ids.min().item() < 0:
        return None

    try:
        out = model(ids, labels=ids)
    except RuntimeError as e:
        # One bad row shouldn't kill the whole run
        print(f"    runtime error on len={ids.size(1)}: {str(e)[:120]}")
        return None

    if not torch.isfinite(out.loss):
        return None

    n_tokens = ids.size(1) - 1
    return -out.loss.item() * n_tokens


def multiblimp_accuracy(model, tok, ds, good_col="sen", bad_col="wrong_sen", limit=None):
    cols = ds.column_names
    if good_col not in cols or bad_col not in cols:
        for g, b in [("sen", "wrong_sen"), ("sentence_good", "sentence_bad"), ("good", "bad")]:
            if g in cols and b in cols:
                good_col, bad_col = g, b
                break
        else:
            raise ValueError(f"Can't find good/bad columns in {cols}")

    n = len(ds) if limit is None else min(limit, len(ds))
    wins, scored, skipped = 0, 0, 0
    for i in range(n):
        g = sentence_logprob(model, tok, ds[i][good_col])
        b = sentence_logprob(model, tok, ds[i][bad_col])
        if g is None or b is None:
            skipped += 1
            continue
        if g > b:
            wins += 1
        scored += 1
    if skipped:
        print(f"    skipped {skipped}/{n} pairs (empty, OOV, too long, or NaN)")
    return wins / scored if scored else float("nan"), scored, skipped


blimp_results = []
for v_name, v_path in VARIANT_PATHS.items():
    print(f"\n=== MultiBLiMP: {v_name} ===")
    try:
        model, tok = load_variant(v_path)
    except Exception as e:
        print(f"  Failed to load model: {e}")
        continue

    for short_lang, iso_lang in MBLIMP_LANGS.items():
        try:
            ds = load_multiblimp_subset(iso_lang)
            acc, scored, skipped = multiblimp_accuracy(model, tok, ds, limit=1000)
            blimp_results.append({
                "variant": v_name,
                "lang": short_lang,
                "MultiBLiMP_acc": acc,
                "n_scored": scored,
                "n_skipped": skipped,
            })
            print(f"  {short_lang}: acc={acc:.3f}  (scored={scored}, skipped={skipped})")
        except Exception as e:
            print(f"  {short_lang}: FAILED ({type(e).__name__}: {str(e)[:150]})")
            blimp_results.append({
                "variant": v_name,
                "lang": short_lang,
                "MultiBLiMP_acc": None,
                "n_scored": 0,
                "n_skipped": 0,
            })

    del model, tok
    # empty_cache() is fine now because CUDA_LAUNCH_BLOCKING made any prior
    # error synchronous; if sentence_logprob returned cleanly, the context
    # is not poisoned
    torch.cuda.empty_cache()

df_blimp = pd.DataFrame(blimp_results)
print("\n", df_blimp)
df_blimp.to_csv(f"{PROJECT_DIR}/results/multiblimp.csv", index=False)


=== MultiBLiMP: A_eng_to_fas ===


Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

data.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/770 [00:00<?, ? examples/s]

...

data.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2553 [00:00<?, ? examples/s]

  [fas] 2553 pairs | columns: ['sen', 'verb', 'verb_idx', 'cop', 'cop_idx', 'child']...
  fa: acc=0.739  (scored=1000, skipped=0)

=== MultiBLiMP: C_joint ===


Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

  [eng] 770 pairs | columns: ['sen', 'verb', 'verb_idx', 'cop', 'cop_idx', 'child']...
  en: acc=0.834  (scored=770, skipped=0)
  [fas] 2553 pairs | columns: ['sen', 'verb', 'verb_idx', 'cop', 'cop_idx', 'child']...
  fa: acc=0.779  (scored=1000, skipped=0)

         variant lang  MultiBLiMP_acc  n_scored  n_skipped
0  A_eng_to_fas   en        0.662338       770          0
1  A_eng_to_fas   fa        0.739000      1000          0
2       C_joint   en        0.833766       770          0
3       C_joint   fa        0.779000      1000          0
